In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd, numpy as np
session = get_active_session()


df = session.sql("""
    SELECT t.* EXCLUDE (LABEL),
           n.NEW_LABEL AS LABEL
    FROM AWS_DP_PROD.ML_OPS.LEADIQ_TRAINING_SET_V4 t
    JOIN (SELECT LEAD_ID, MAX(TARGET_CONVERTED) AS NEW_LABEL
          FROM AWS_DP_PROD.ML_OPS.LEADIQ_TARGET_LABELS_V4
          GROUP BY LEAD_ID) n USING (LEAD_ID)          -- 去重:247 个重复 LEAD_ID
""").to_pandas()

drop_cols = ['LEAD_ID', 'LEAD_TS', 'LABEL', 'BOAT_LEVEL_POS']
features  = [c for c in df.columns if c not in drop_cols]
print(len(features), 'features')          # 24

df['LEAD_TS'] = pd.to_datetime(df['LEAD_TS'], utc=True).dt.tz_localize(None)
train_end  = pd.Timestamp('2024-04-30')                     
test_start = pd.Timestamp('2024-08-28')                       
test_end   = df['LEAD_TS'].max() - pd.DateOffset(months=15)   

tr = df[df['LEAD_TS'] <= train_end]
te = df[(df['LEAD_TS'] >= test_start) & (df['LEAD_TS'] <= test_end)]
X_tr, y_tr = tr[features], tr['LABEL'].values
X_te, y_te = te[features], te['LABEL'].values

spw = (y_tr == 0).sum() / (y_tr == 1).sum()   # 自动重算,预期 ~40-45(不再是283)
print(f'train {len(tr):,}  test {len(te):,}  test pos rate {y_te.mean():.4f}  spw {spw:.0f}')


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

def lift_at_k(y, p, k):
    n = max(int(len(y) * k), 1)
    top = np.argsort(-p)[:n]
    return y[top].mean() / y.mean()

params = dict(n_estimators=300, max_depth=6, learning_rate=0.1,
              scale_pos_weight=spw, eval_metric='aucpr',
              missing=np.nan, random_state=42, n_jobs=-1)   #

groups = {
    'FULL':              [],
    'minus_GA4':         ['VIEWED_THIS_BOAT','CLICKED_GALLERY','SAVED_BOAT','IS_NEW_VISITOR'],
    'minus_EMAIL_HIST':  ['N_PRIOR_LEADS','N_DISTINCT_LISTINGS_PRIOR',
                          'DAYS_SINCE_LAST_LEAD','PRIOR_BOAT_OWNER'],
    'minus_PIPELINE':    [c for c in features if c.startswith('SRC_')],
    'minus_GA4_and_PIPE':['VIEWED_THIS_BOAT','CLICKED_GALLERY','SAVED_BOAT','IS_NEW_VISITOR']
                         + [c for c in features if c.startswith('SRC_')],
    'minus_GEO':         ['SAME_STATE','CROSS_COUNTRY','USER_LISTING_DISTANCE'],
    'minus_PRICE':       ['LISTING_PRICE'],
}

rows = []
for name, drop in groups.items():
    feats = [f for f in features if f not in drop]
    m = XGBClassifier(**params).fit(X_tr[feats], y_tr)
    p = m.predict_proba(X_te[feats])[:, 1]
    rows.append({'run': name, 'n_feats': len(feats),
                 'lift@1%': round(lift_at_k(y_te, p, 0.01), 2),
                 'lift@5%': round(lift_at_k(y_te, p, 0.05), 2),
                 'lift@10%': round(lift_at_k(y_te, p, 0.10), 2),
                 'roc_auc': round(roc_auc_score(y_te, p), 4)})
    print(rows[-1])

results = pd.DataFrame(rows)
results